# 03. Tool Calling 실습
**SK하이닉스 Autonomous R&D — Day 3**

> 목표: OpenAI의 Tool Calling

---
## 0. 라이브러리 설치

In [ ]:
!pip install openai python-dotenv pytz yfinance

In [ ]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('✅ 준비 완료')

---
## 1. Tool Calling 흐름 이해

```
사용자 질문
    ↓
LLM이 "도구가 필요함" 판단
    ↓
Tool 호출 요청(name + arguments)
    ↓
Python 함수 실행 후 결과 반환
    ↓
LLM이 결과를 자연어로 정리해 최종 답변
```

핵심은 2가지입니다.
- LLM은 **직접 조회/계산**하지 않고, 필요한 함수를 고릅니다.
- 함수의 입력/출력은 **JSON 스키마와 문자열(JSON)**로 주고받습니다.

In [ ]:
question = '지금 현재 시간이 몇시야?'
r = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': question}],
)
print(f'{r.choices[0].message.content}')

---
## 2. 첫 Tool 만들기 — `datetime`으로 도시 시간 조회(기본편)

- `datetime.now()`를 써서 현재 시간을 만들고
- 도시 이름은 그대로 붙여서 반환합니다.

In [ ]:
from datetime import datetime

def get_city_time_basic():
    """도시 현재 시간을 반환(기본 버전: 시간대 미반영)"""
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return now

print(get_city_time_basic())

---
## 3. Tool 스키마 등록 (기본 `datetime` Tool)

In [ ]:
basic_tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_city_time_basic',
            'description': '현재 시간을 반환합니다.'
            },
    
    }
]

BASIC_FUNCTIONS = {
    'get_city_time_basic': get_city_time_basic,
}

---
## 4. Tool 호출 

In [ ]:
messages = [
    {
        'role': 'system',
        'content': 'You are a helpful assistant. Use tools when needed. 답변은 한국어로 해.',
    },
    {'role': 'user', 'content': '지금 서울이 몇 시인지 알려줘'},
]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=messages,
    tools=basic_tools, # 여기에 tool list 입력
)

msg = response.choices[0].message
print('tool_calls:', msg.tool_calls)

In [ ]:
# 1) assistant의 tool call 메시지를 대화 이력에 추가
messages.append(msg)

# 2) 각 tool을 파이썬에서 실행하고 결과를 role='tool'로 다시 추가
for tool_call in msg.tool_calls:
    fn_name = tool_call.function.name
    fn_args = json.loads(tool_call.function.arguments)

    print(f'▶ 호출 함수: {fn_name}')
    print(f'  인자: {fn_args}')

    result = BASIC_FUNCTIONS[fn_name](**fn_args)
    print(f'  실행 결과: {result}')

    messages.append(
        {
            'role': 'tool',
            'tool_call_id': tool_call.id,
            'content': result,
        }
    )

# 3) tool 결과를 반영한 최종 답변 생성
final = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=messages,
    tools=basic_tools,
)
print('\n=== 최종 답변 ===')
print(final.choices[0].message.content)


---
## 5. 다른 도시 시간 확인

In [ ]:
def run_agent_once(user_question, tools, tool_functions, system_msg):
    messages = [
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_question},
    ]

    first = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=messages, tools=tools
    )
    msg = first.choices[0].message

    if not msg.tool_calls:
        return msg.content

    messages.append(msg)
    for tc in msg.tool_calls:
        fn = tc.function.name
        args = json.loads(tc.function.arguments or '{}')
        result = tool_functions[fn](**args)
        print(f'  {fn}({args}) -> {result}')
        messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

    final = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=messages, tools=tools
    )
    return final.choices[0].message.content

In [ ]:
print('=== (1) 기본 datetime Tool 한계 확인 ===')
print(run_agent_once(
    '뉴욕 현재 시각 알려줘.',
    basic_tools,
    BASIC_FUNCTIONS,
    'Use tools when needed. 답변은 한국어로 해.',
))

In [ ]:
import pytz

# 개선 버전: pytz로 도시별 시간대 정확히 계산
CITY_TZ = {
    '서울': 'Asia/Seoul',
    'seoul': 'Asia/Seoul',
    '뉴욕': 'America/New_York',
    'new york': 'America/New_York',
    '도쿄': 'Asia/Tokyo',
    'tokyo': 'Asia/Tokyo',
    '런던': 'Europe/London',
    'london': 'Europe/London',
}


def get_city_time_tz(city: str) -> str:
    key = city.strip().lower()
    tz_name = CITY_TZ.get(key)
    if not tz_name:
        return json.dumps({'error': f'지원하지 않는 도시: {city}'}, ensure_ascii=False)

    now = datetime.now(pytz.timezone(tz_name)).strftime('%Y-%m-%d %H:%M:%S')
    return json.dumps({'city': city, 'timezone': tz_name, 'current_time': now}, ensure_ascii=False)


tz_tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_city_time_tz',
            'description': '도시의 시간대를 반영해 현재 시간을 반환합니다.',
            'parameters': {
                'type': 'object',
                'properties': {'city': {'type': 'string'}},
                'required': ['city'],
            },
        },
    }
]

TZ_FUNCTIONS = {'get_city_time_tz': get_city_time_tz}

In [ ]:
print('\n=== (2) pytz 적용 후 ===')
print(run_agent_once(
    '서울과 뉴욕의 현재 시각을 각각 알려줘.',
    tz_tools,
    TZ_FUNCTIONS,
    'Use tools when needed. 답변은 한국어로 해.',
))

## 실습1 : 주식 조회

In [ ]:
# 확장 버전: yfinance로 미국 주식 조회
import yfinance as yf

def get_us_stock_price(ticker):
    symbol = ticker.strip().upper()
    try:
        t = yf.Ticker(symbol)
        hist = t.history(period='2d')
        if hist.empty:
            return json.dumps({'error': f'{symbol} 데이터가 없습니다.'}, ensure_ascii=False)

        latest = hist.iloc[-1]
        prev_close = float(latest['Close']) if 'Close' in latest else None
        open_price = float(latest['Open']) if 'Open' in latest else None

        return json.dumps(
            {
                'ticker': symbol,
                'open': round(open_price, 2) if open_price is not None else None,
                'close': round(prev_close, 2) if prev_close is not None else None,
                'currency': 'USD',
                'source': 'yfinance',
            },
            ensure_ascii=False,
        )
    except Exception as e:
        return json.dumps({'error': str(e)}, ensure_ascii=False)


In [ ]:
stock_tools = [
    {
        'type': 'function',
        'function': {
            'name': '', ## 입력
            'description': '', ## 입력
            'parameters': {
                'type': 'object',
                'properties': {'ticker': {'type': 'string'}},
                'required': ['ticker'],
            },
        },
    }
]

STOCK_FUNCTIONS = {'get_us_stock_price': get_us_stock_price}

print('\n=== (3) yfinance 주식 Tool ===')
print(run_agent_once(
    '애플(AAPL)과 테슬라(TSLA) 최근 가격을 알려주고 간단 코멘트해줘.',
    stock_tools,
    STOCK_FUNCTIONS,
    'Use tools when needed. 한국어로 간결하게 답변해.',
))

## 실습 2: 위의 tool을 포함한 chatbot 생성